# Router experiments on the 100k-v2 rung — train, validate, compare with v3

The same harness as `route_experiments.ipynb` / `route_experiments_v3.ipynb`,
pointed at the **100k-v2 rung**'s labelled rows
(`data/rungs/100k-v2/labeling/labels.parquet`, 91,093 rows). Features come from
`catalog_v3` by inner join, exactly as the v3 notebook pairs v2 labels with the
v3 catalog; the join keeps the ~90.8% of rows the catalog covers (the rest are
the rung's augmented / doc_grounded rows neither older catalog carries). Edit
the **config** cell, Run All. Section 8 trains the 100k-v2 and v3 routers under
identical config and cross-evaluates them on each other's held-out rows.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

from hybrid_search_rrf_dataset.router import (
    Representation,
    RouterExperiment,
    StrategyRouter,
)

DATA = Path("data")
LABELS_100K_V2 = DATA / "rungs" / "100k-v2" / "labeling" / "labels.parquet"
V3_CATALOG = DATA / "v3" / "catalog_v3.parquet"
V3_LABELS = DATA / "v3" / "dataset_v3.parquet"

# ---- config: edit and Run All ----
REPRESENTATION = Representation.ENGINEERED
ENCODER = None
PROTOCOL = 'random_within_lane'   # or 'holdout_lane'
ALL_ROWS = 'decisive'   # 'decisive' = clear winners | 'recommended' | 'all'
MAX_CLASS_SHARE = None

exp = RouterExperiment(labels_path=LABELS_100K_V2, catalog_path=V3_CATALOG,
                       encoder=ENCODER)

/Users/andrei/projects/hybrid-search-rrf-dataset/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 1 — train on 100k-v2
from hybrid_search_rrf_dataset.router import _derive_engineered, _margin, _routes_differ

train, test = exp.split(PROTOCOL)
router = StrategyRouter(REPRESENTATION, encoder=ENCODER).fit(
    train, all_rows=ALL_ROWS, max_class_share=MAX_CLASS_SHARE
)
print('thresholds (dense, sparse):', tuple(round(t, 3) for t in router.thresholds))

mode = {False: 'decisive', True: 'recommended'}.get(ALL_ROWS, ALL_ROWS)
fit_rows = (
    train if mode == 'all'
    else train[_routes_differ(train)] if mode == 'recommended'
    else train[_margin(train) >= router.decisive_margin]
)
eng = _derive_engineered(fit_rows)
coef = router.coefficients().set_index('feature')
coef['fires'] = [(eng[f] > 0).sum() if f in eng.columns else None for f in coef.index]
coef.round(3).nlargest(8, 'sparse_weight')

thresholds (dense, sparse): (0.5, 0.5)


,dense_weight,sparse_weight,fires
feature,,,
query_corpus.max_idf,-0.788,0.951,4119
query_corpus.vocab_overlap,-0.607,0.597,4119
length.length_chars,-1.024,0.527,5159
query_corpus.avg_idf,-0.380,0.429,4119
structured_identifiers.error_code_like,-0.147,0.327,23
structured_identifiers.bic,-0.447,0.322,349
query_corpus.min_pmi,-0.196,0.208,782
term_rarity.rare_share,-0.158,0.202,4819


In [4]:
# 2 — validate: the six-column headroom table on 100k-v2
cols = [
    'protocol', 'representation', 'all_rows', 'max_class_share', 'n_test_decisive',
    'const_dense_only', 'const_pure_rrf', 'const_sparse_only',
    'oracle', 'router', 'headroom_captured', 't_dense', 't_sparse',
]
result = exp.run(
    representations=[REPRESENTATION],
    all_rows=ALL_ROWS,
    max_class_share=MAX_CLASS_SHARE,
)
result[cols].round(3)

router ablation:   0%|          | 0/2 [00:00<?, ?it/s]

random_within_lane·engineered:   0%|          | 0/2 [00:00<?, ?it/s]

random_within_lane·engineered:   0%|          | 0/2 [00:00<?, ?it/s, fitting]

random_within_lane·engineered:   0%|          | 0/2 [00:00<?, ?it/s, tuning thresholds]

random_within_lane·engineered:   0%|          | 0/2 [00:00<?, ?it/s, evaluating]       

...

holdout_lane·engineered:  50%|█████     | 1/2 [00:00<00:00,  3.21it/s, evaluating]       

holdout_lane·engineered:  50%|█████     | 1/2 [00:00<00:00,  3.21it/s, headroom=-0.088, n=652]

holdout_lane·engineered: 100%|██████████| 2/2 [00:00<00:00,  3.51it/s, headroom=-0.088, n=652]

holdout_lane·engineered: 100%|██████████| 2/2 [00:00<00:00,  3.46it/s, headroom=-0.088, n=652]

,protocol,representation,all_rows,max_class_share,n_test_decisive,const_dense_only,const_pure_rrf,const_sparse_only,oracle,router,headroom_captured,t_dense,t_sparse
0,random_within_lane,engineered,decisive,None,1657,0.456,0.214,0.539,0.978,0.736,0.449,0.10,0.5
1,holdout_lane,engineered,decisive,None,652,0.454,0.247,0.534,1.000,0.493,-0.088,0.55,0.1


In [5]:
# 3 — use: route your own queries with the section-1 router
my_queries = [
    'Who likes Curling?',
    'what are the side effects of DHA',
    'CVE-2021-44228 log4j remote code execution',
    'http://localhost.com',
]
for q in my_queries:
    e = router.explain(q)
    print(f"{e['route']!s:12s} p_dense={e['p_dense']:.2f} p_sparse={e['p_sparse']:.2f}  {q}")

sparse_only  p_dense=0.42 p_sparse=0.53  Who likes Curling?
dense_only   p_dense=0.52 p_sparse=0.44  what are the side effects of DHA
sparse_only  p_dense=0.15 p_sparse=0.83  CVE-2021-44228 log4j remote code execution
sparse_only  p_dense=0.27 p_sparse=0.61  http://localhost.com


In [6]:
# 4 — serve: refit on ALL 100k-v2 rows (not comparable to section 2's held-out numbers)
data = exp.load()
served = StrategyRouter(REPRESENTATION, encoder=ENCODER, delta=0.05).fit(
    data, all_rows='all', max_class_share=MAX_CLASS_SHARE
)
served.tune_thresholds(data)
print('serving thresholds (dense, sparse):', tuple(round(t, 3) for t in served.thresholds))
for q in my_queries:
    e = served.explain(q)
    print(f"{e['route']!s:12s} p_dense={e['p_dense']:.2f} p_sparse={e['p_sparse']:.2f}  {q}")

serving thresholds (dense, sparse): (0.1, 0.4)
pure_rrf     p_dense=0.53 p_sparse=0.52  Who likes Curling?
dense_only   p_dense=0.60 p_sparse=0.48  what are the side effects of DHA
sparse_only  p_dense=0.33 p_sparse=0.66  CVE-2021-44228 log4j remote code execution


sparse_only  p_dense=0.61 p_sparse=0.67  http://localhost.com


In [7]:
# 5 — margin hedge on 100k-v2 (delta tuned on train, judged held-out)
import numpy as np

from hybrid_search_rrf_dataset.router import _mean_objective, _route_from_probs

tune_frame = train[_routes_differ(train)]
p_d, p_s = router._probabilities(tune_frame)
deltas = np.round(np.arange(0.0, 0.32, 0.002), 2)[::-1]
tune_scores = [
    _mean_objective(tune_frame, _route_from_probs(p_d, p_s, 0.5, 0.5, delta=d))
    for d in deltas
]
best_delta = float(deltas[int(np.argmax(tune_scores))])
print(f'best delta on train: {best_delta:.2f}')

decisive_test = test[_margin(test) >= router.decisive_margin]
pt_d, pt_s = router._probabilities(decisive_test)
for label, (td, ts, d) in {
    'tuned thresholds, delta=0': (*router.thresholds, 0.0),
    'symmetric (0.5, 0.5), delta=0': (0.5, 0.5, 0.0),
    f'symmetric + delta={best_delta:.2f}': (0.5, 0.5, best_delta),
}.items():
    routes = _route_from_probs(pt_d, pt_s, td, ts, delta=d)
    score = _mean_objective(decisive_test, routes)
    n_rrf = sum(r.value == 'pure_rrf' for r in routes)
    print(f'{label:32s} held-out: {score:.3f}  (rrf fired on {n_rrf}/{len(routes)})')

best delta on train: 0.00
tuned thresholds, delta=0        held-out: 0.735  (rrf fired on 31/1657)
symmetric (0.5, 0.5), delta=0    held-out: 0.735  (rrf fired on 31/1657)
symmetric + delta=0.00           held-out: 0.735  (rrf fired on 31/1657)


# 6 — Acceptability heads on 100k-v2 (SPEC d60 form)

Three binary heads on `ok_*`, cheapest acceptable route serves. The 100k-v2
rung carries a large tied mass (27.2% all_tied), so watch the cost advantage
of the heads router over argmax here.

In [8]:
import numpy as np
import pandas as pd

from hybrid_search_rrf_dataset.fusion import SERVING_COST, StrategyName
from hybrid_search_rrf_dataset.labels import AcceptabilityLabels
from hybrid_search_rrf_dataset.router import AcceptabilityRouter, _mean_objective, _margin

acc = AcceptabilityRouter(REPRESENTATION, encoder=ENCODER).fit(train)
view = AcceptabilityLabels(test).frame()
answerable = view[view["serve"].notna()].reset_index(drop=True)
print(f"test: {len(answerable):,} answerable of {len(view):,}")

acc_routes = acc.predict_routes(answerable)
argmax_routes = router.predict_routes(answerable)
decisive_mask = (_margin(answerable) >= router.decisive_margin).to_numpy()
serve_oracle = [StrategyName(s) for s in answerable["serve"]]

def readout(policy, routes):
    routes = list(routes)
    mix = pd.Series([r.value for r in routes]).value_counts(normalize=True)
    return {
        "policy": policy,
        "objective (answerable)": _mean_objective(answerable, routes),
        "objective (decisive)": _mean_objective(
            answerable[decisive_mask], [r for r, m in zip(routes, decisive_mask) if m]),
        "serve agreement": float(np.mean([r == s for r, s in zip(routes, serve_oracle)])),
        "mean cost": float(np.mean([SERVING_COST[r] for r in routes])),
        **{f"% {s.value}": float(mix.get(s.value, 0.0)) for s in StrategyName},
    }

pd.DataFrame([
    readout("serve oracle (view)", serve_oracle),
    readout("acceptability heads", acc_routes),
    readout("argmax router (section 1)", argmax_routes),
    *(readout(f"const {s.value}", [s] * len(answerable)) for s in StrategyName),
]).round(3)

test: 12,628 answerable of 16,549


,policy,objective (answerable),objective (decisive),serve agreement,mean cost,% dense_only,% pure_rrf,% sparse_only
0,serve oracle (view),0.753,0.978,1.000,0.171,0.159,0.006,0.835
1,acceptability heads,0.676,0.732,0.542,0.549,0.504,0.022,0.473
2,argmax router (section 1),0.675,0.735,0.547,0.540,0.500,0.020,0.480
3,const dense_only,0.590,0.456,0.159,1.000,1.000,0.000,0.000
4,const pure_rrf,0.656,0.214,0.006,2.000,0.000,1.000,0.000
5,const sparse_only,0.608,0.539,0.835,0.000,0.000,0.000,1.000


In [9]:
# 7 — archetype probes: mechanics-anchored fixed instrument
from hybrid_search_rrf_dataset.probes import probe

argmax_probe, heads_probe = probe(served), probe(acc)
compare = argmax_probe[["query", "expected"]].copy()
compare["argmax route"] = argmax_probe["served"]
compare["argmax ok"] = argmax_probe["agrees"]
compare["heads route"] = heads_probe["served"]
compare["heads ok"] = heads_probe["agrees"]
for name, frame in (("argmax", argmax_probe), ("heads", heads_probe)):
    decided = frame["agrees"].notna().sum()
    print(f"{name:7s} agrees with mechanics on {frame['agrees'].eq(True).sum()}/{decided}")
compare

argmax  agrees with mechanics on 1/6
heads   agrees with mechanics on 2/6


,query,expected,argmax route,argmax ok,heads route,heads ok
0,a3f5d8b9e12c4d56789abcdef0123456,sparse_only,dense_only,False,sparse_only,True
1,/etc/nginx/nginx.conf,sparse_only,sparse_only,True,sparse_only,True
2,ERR_CONNECTION_RESET,sparse_only,dense_only,False,dense_only,False
3,explain quicksort,dense_only,sparse_only,False,sparse_only,False
4,HTTP 502,NaN,sparse_only,<NA>,sparse_only,<NA>
5,comment volent les oiseaux,dense_only,sparse_only,False,sparse_only,False
6,como aprender a programar en rust,dense_only,sparse_only,False,sparse_only,False


# 8 — v2 vs v3 vs 100k-v2, identical features

All three sources use catalog_v3 features, so the representation is identical —
only the training data changes. Two honest readings: (a) each source's router
against ITS OWN constants and oracle (`headroom_captured`, the number that
survives cross-source comparison because it divides out each source's floor and
ceiling); (b) the transfer matrix — train on one, evaluate on another's held-out
decisive rows. Within-lane, 100k-v2 leads.

In [10]:
V2_LABELS = DATA / "route_labels" / "labels.parquet"
v2_exp = RouterExperiment(labels_path=V2_LABELS, catalog_path=V3_CATALOG, encoder=ENCODER)
v3_exp = RouterExperiment(labels_path=V3_LABELS, catalog_path=V3_CATALOG, encoder=ENCODER)

side = []
for name, e in (("v2", v2_exp), ("v3", v3_exp), ("100k-v2", exp)):
    r = e.run(representations=[REPRESENTATION], all_rows=ALL_ROWS,
              max_class_share=MAX_CLASS_SHARE)
    r.insert(0, "source", name)
    side.append(r[["source", *cols]])
# headroom_captured is the honest ranking column; within-lane 100k-v2 leads
pd.concat(side, ignore_index=True).round(3)

router ablation:   0%|          | 0/2 [00:00<?, ?it/s]

random_within_lane·engineered:   0%|          | 0/2 [00:00<?, ?it/s]

random_within_lane·engineered:   0%|          | 0/2 [00:00<?, ?it/s, fitting]

random_within_lane·engineered:   0%|          | 0/2 [00:00<?, ?it/s, tuning thresholds]

random_within_lane·engineered:   0%|          | 0/2 [00:00<?, ?it/s, evaluating]       

...

holdout_lane·engineered:  50%|█████     | 1/2 [00:00<00:00,  5.48it/s, evaluating]       

holdout_lane·engineered:  50%|█████     | 1/2 [00:00<00:00,  5.48it/s, headroom=-0.088, n=652]

holdout_lane·engineered: 100%|██████████| 2/2 [00:00<00:00,  5.20it/s, headroom=-0.088, n=652]

holdout_lane·engineered: 100%|██████████| 2/2 [00:00<00:00,  5.24it/s, headroom=-0.088, n=652]

,source,protocol,representation,all_rows,max_class_share,n_test_decisive,const_dense_only,const_pure_rrf,const_sparse_only,oracle,router,headroom_captured,t_dense,t_sparse
0,v2,random_within_lane,engineered,decisive,None,1021,0.631,0.208,0.367,0.975,0.765,0.391,0.10,0.65
1,v2,holdout_lane,engineered,decisive,None,206,0.607,0.185,0.430,1.000,0.655,0.121,0.30,0.65
2,v3,random_within_lane,engineered,decisive,None,1961,0.504,0.244,0.500,0.992,0.586,0.169,0.55,0.30
3,v3,holdout_lane,engineered,decisive,None,463,0.436,0.268,0.538,1.000,0.490,-0.103,0.55,0.40
4,100k-v2,random_within_lane,engineered,decisive,None,1657,0.456,0.214,0.539,0.978,0.736,0.449,0.10,0.50
5,100k-v2,holdout_lane,engineered,decisive,None,652,0.454,0.247,0.534,1.000,0.493,-0.088,0.55,0.10


In [11]:
# transfer matrix: rows = trained on, columns = evaluated on (held-out decisive)
import pandas as pd
from hybrid_search_rrf_dataset.fusion import StrategyName
from hybrid_search_rrf_dataset.router import _margin, _mean_objective

splits = {"v2": v2_exp.split(PROTOCOL), "v3": v3_exp.split(PROTOCOL),
          "100k-v2": exp.split(PROTOCOL)}
routers = {n: StrategyRouter(REPRESENTATION, encoder=ENCODER).fit(tr, all_rows=ALL_ROWS)
           for n, (tr, _te) in splits.items()}
tests = {f"{n} test": te for n, (_tr, te) in splits.items()}

def decisive(frame, r):
    return frame[_margin(frame) >= r.decisive_margin]

def oracle_obj(dec):
    idx = dec[["score_dense_only", "score_pure_rrf", "score_sparse_only"]] \
        .idxmax(axis=1).str.replace("score_", "")
    return _mean_objective(dec, [StrategyName(s) for s in idx])

matrix = {}
for tn, r in routers.items():
    matrix[f"trained on {tn}"] = {
        en: round(_mean_objective(decisive(te, r), r.predict_routes(decisive(te, r))), 3)
        for en, te in tests.items()}
transfer = pd.DataFrame(matrix).T
anyr = next(iter(routers.values()))
transfer.loc["oracle (ceiling)"] = {en: round(oracle_obj(decisive(te, anyr)), 3)
                                     for en, te in tests.items()}
transfer

,v2 test,v3 test,100k-v2 test
trained on v2,0.751,0.323,0.562
trained on v3,0.507,0.571,0.572
trained on 100k-v2,0.766,0.543,0.735
oracle (ceiling),0.975,0.992,0.978


# 9 — "generalizes better" is a test-set artifact (the holdout caveat)

Section 8's `holdout_lane` numbers are NOT apples-to-apples across sources: each
rung populates the held-out lane (`rarb-math`) with a *different* set of rows,
so a source can look like it generalizes better only because its slice of the
lane is smaller and differently balanced. Below: (a) the two rarb-math test sets
differ — v2's is a dense-favoring subset of 100k-v2's sparse-favoring set; (b)
on a COMMON test set the gap largely reverses; (c) the real residual is
overconfidence under shift — 100k-v2's sharper (better in-lane) boundary stays
committed off-distribution where v2's hedges.

In [12]:
# (a) same lane, different rows — the confound
from hybrid_search_rrf_dataset.router import _winner

HOLD = RouterExperiment.HOLDOUT_LANE   # 'rarb-math'
v2_data, k_data = v2_exp.load(), exp.load()
for name, d in (("v2", v2_data), ("100k-v2", k_data)):
    hm = d[d.dataset == HOLD]
    dec = hm[_margin(hm) >= 0.4]
    print(f"{name:8s} {HOLD}: {len(hm):5,} rows, {len(dec):4,} decisive, "
          f"winner mix {dict(_winner(dec).value_counts())}")
a = set(v2_data.query("dataset==@HOLD").query_id.astype(str))
b = set(k_data.query("dataset==@HOLD").query_id.astype(str))
print(f"\nquery_id overlap: v2={len(a)}, 100k-v2={len(b)}, shared={len(a & b)} "
      f"(v2 {'IS' if a <= b else 'is NOT'} a subset of 100k-v2)")

v2       rarb-math:   440 rows,  206 decisive, winner mix {'dense_only': np.int64(121), 'sparse_only': np.int64(85)}
100k-v2  rarb-math: 6,327 rows,  652 decisive, winner mix {'sparse_only': np.int64(329), 'dense_only': np.int64(274), 'pure_rrf': np.int64(49)}

query_id overlap: v2=440, 100k-v2=6327, shared=440 (v2 IS a subset of 100k-v2)


In [13]:
# (b) fair test: train each on its own non-holdout rows, evaluate BOTH on the
# SAME rarb-math slice. The section-8 ranking reverses on the dense-favoring slice.
rv2 = StrategyRouter(REPRESENTATION, encoder=ENCODER).fit(
    v2_data[v2_data.dataset != HOLD], all_rows="decisive")
rk = StrategyRouter(REPRESENTATION, encoder=ENCODER).fit(
    k_data[k_data.dataset != HOLD], all_rows="decisive")

rows = []
for tname, src in (("v2 slice (dense-fav)", v2_data),
                   ("100k-v2 slice (sparse-fav)", k_data)):
    hm = src[src.dataset == HOLD]
    dec = hm[_margin(hm) >= 0.4]
    rows.append({
        "common test": tname, "n_decisive": len(dec),
        "const_dense": round(_mean_objective(dec, [StrategyName.DENSE_ONLY] * len(dec)), 3),
        "const_sparse": round(_mean_objective(dec, [StrategyName.SPARSE_ONLY] * len(dec)), 3),
        "oracle": round(oracle_obj(dec), 3),
        "v2-trained": round(_mean_objective(dec, rv2.predict_routes(dec)), 3),
        "100k-v2-trained": round(_mean_objective(dec, rk.predict_routes(dec)), 3),
    })
pd.DataFrame(rows)

,common test,n_decisive,const_dense,const_sparse,oracle,v2-trained,100k-v2-trained
0,v2 slice (dense-fav),206,0.607,0.430,1.0,0.602,0.645
1,100k-v2 slice (sparse-fav),652,0.454,0.534,1.0,0.509,0.489


In [14]:
# (c) the mechanism: confidence under distribution shift. 100k-v2's sharper
# boundary (its in-lane asset) barely backs off on the unseen lane, so it commits
# hard -- and confidently wrong (over-routes dense where truth is sparse-majority).
import numpy as np

hm = k_data[k_data.dataset == HOLD]
dec = hm[_margin(hm) >= 0.4]
true_mix = {kk: round(vv, 3) for kk, vv in _winner(dec).value_counts(normalize=True).items()}
print(f"true winner mix on unseen {HOLD}: {true_mix}\n")
for rn, r in (("v2-trained", rv2), ("100k-v2-trained", rk)):
    p_d, p_s = r._probabilities(dec)
    gap = np.abs(np.asarray(p_d) - np.asarray(p_s))
    mix = pd.Series([x.value for x in r.predict_routes(dec)]).value_counts(normalize=True)
    print(f"{rn:16s} commitment mean|p_dense-p_sparse|={gap.mean():.3f} "
          f"frac>0.3={np.mean(gap > 0.3):.3f} | predicts dense={mix.get('dense_only', 0):.2f}")

true winner mix on unseen rarb-math: {'sparse_only': 0.505, 'dense_only': 0.42, 'pure_rrf': 0.075}

v2-trained       commitment mean|p_dense-p_sparse|=0.277 frac>0.3=0.382 | predicts dense=0.60
100k-v2-trained  commitment mean|p_dense-p_sparse|=0.445 frac>0.3=0.706 | predicts dense=0.87
